# 01 - Bronze Bookings Raw Refresh

Sanitized portfolio notebook for the Corporate Travel Booking Data Platform.

Before running this notebook in a real Databricks workspace, replace placeholder paths such as `<storage-account>` and confirm that the Unity Catalog schemas exist.
All outputs have been cleared before publishing.


In [ ]:
source_path = "abfss://raw-booking-data@<storage-account>.dfs.core.windows.net/bookings/"


In [ ]:
checkpoint_path = "abfss://checkpoints@<storage-account>.dfs.core.windows.net/bronze/bookings_raw/"


In [ ]:
schema_path = "abfss://schemas@<storage-account>.dfs.core.windows.net/bronze/bookings_raw/"


In [ ]:
target_table = "corporate_travel_analytics.bronze.bookings_raw"


In [ ]:
from pyspark.sql.functions import current_timestamp, col, lit

df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", schema_path) \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .load(source_path) \
    .withColumn("_corrupt_record", col("_rescued_data")) \
    .drop("_rescued_data") \
    .withColumn("ingest_time", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("record_type", lit("booking")) \
    .select(
        "BookingCreatedAt",
        "_corrupt_record",
        "_id",
        "updatedAt",
        "rawStoredAt",
        "rawEventId",
        "approval",
        "bookingId",
        "bookingStatus",
        "costDetails",
        "employee",
        "hotelDetails",
        "pocEmail",
        "travelDetails",
        "ingest_time",
        "source_file",
        "record_type"
    )


In [ ]:
df.writeStream \
    .format("delta") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .toTable(target_table)
